In [ ]:
# ==========================================================
# NOTEBOOK 02
# DATA PREPROCESSING
# ==========================================================

"""
OBJECTIVE
---------
This notebook prepares the COVID-19 Chest X-ray dataset for model training.

The following tasks are performed:
1. Mount Google Drive
2. Extract Dataset
3. Create a Balanced Dataset
4. Split Dataset
5. Encode Labels
6. Compute Class Weights
7. Create TensorFlow Data Pipelines
8. Save Processed Files

Output:
-------
train.csv
validation.csv
test.csv
label_encoder.pkl
class_weights.npy
"""

# ==========================================================
# IMPORT LIBRARIES
# ==========================================================

import os
import random
import shutil
import zipfile
import pickle

import numpy as np
import pandas as pd
import tensorflow as tf

from google.colab import drive

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight

# ==========================================================
# PROJECT SETTINGS
# ==========================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

IMAGE_SIZE = (224,224)

BATCH_SIZE = 32

print("="*60)
print("TensorFlow Version :", tf.__version__)
print("GPU Available :", tf.config.list_physical_devices("GPU"))
print("="*60)

# ==========================================================
# MOUNT GOOGLE DRIVE
# ==========================================================

drive.mount("/content/drive")

# ==========================================================
# PROJECT PATHS
# ==========================================================

PROJECT_PATH = "/content/drive/MyDrive/Covid_Major_Project"

ZIP_FILE = os.path.join(
    PROJECT_PATH,
    "dataset",
    "archive.zip"
)

TEMP_FOLDER = "/content/COVID19_Dataset"

# ==========================================================
# REMOVE OLD EXTRACTION
# ==========================================================

if os.path.exists(TEMP_FOLDER):
    shutil.rmtree(TEMP_FOLDER)

os.makedirs(TEMP_FOLDER, exist_ok=True)

# ==========================================================
# EXTRACT DATASET
# ==========================================================

print("\nExtracting Dataset...")
print("Please wait...")

with zipfile.ZipFile(ZIP_FILE,"r") as zip_ref:
    zip_ref.extractall(TEMP_FOLDER)

print("\nDataset Extracted Successfully.")

DATASET_PATH = os.path.join(
    TEMP_FOLDER,
    "COVID-19_Radiography_Dataset"
)

print(DATASET_PATH)

# ==========================================================
# CREATE A BALANCED DATASET
# ==========================================================

"""
To reduce training time while maintaining fairness across all classes,
1,000 images are randomly selected from each class.
"""

LIMITS = {
    "COVID": 1000,
    "Normal": 1000,
    "Lung_Opacity": 1000,
    "Viral Pneumonia": 1000
}

image_paths = []
labels = []

print("="*60)
print("CREATING BALANCED DATASET")
print("="*60)

for class_name, max_images in LIMITS.items():

    image_folder = os.path.join(
        DATASET_PATH,
        class_name,
        "images"
    )

    image_files = [
        file
        for file in os.listdir(image_folder)
        if file.lower().endswith(".png")
    ]

    random.shuffle(image_files)

    image_files = image_files[:max_images]

    print(f"{class_name:20s} : {len(image_files)} images selected")

    for file in image_files:

        image_paths.append(
            os.path.join(image_folder, file)
        )

        labels.append(class_name)

# ==========================================================
# CREATE DATAFRAME
# ==========================================================

df = pd.DataFrame({

    "Image": image_paths,

    "Label": labels

})

print("\nDataset Created Successfully.\n")

print(df.head())

print("\nClass Distribution\n")

print(df["Label"].value_counts())

print("\nTotal Images :", len(df))

# ==========================================================
# SHUFFLE THE DATASET
# ==========================================================

df = df.sample(

    frac=1,

    random_state=SEED

).reset_index(drop=True)

print("\nDataset Shuffled Successfully.")

print(df.head())

# ==========================================================
# VERIFY IMAGE PATHS
# ==========================================================

print("="*60)
print("VERIFYING IMAGE PATHS")
print("="*60)

missing = 0

for img in df["Image"]:

    if not os.path.exists(img):
        missing += 1

print("Missing Images :", missing)

print("Available Images :", len(df)-missing)

assert missing == 0

print("\nDataset Verification Successful.")

# ==========================================================
# SPLIT DATASET
# ==========================================================

"""
Dataset Split

Training    : 70%
Validation  : 15%
Testing      : 15%

Stratified splitting is used so that all classes are
equally represented in each subset.
"""

train_df, temp_df = train_test_split(

    df,

    test_size=0.30,

    stratify=df["Label"],

    random_state=SEED

)

val_df, test_df = train_test_split(

    temp_df,

    test_size=0.50,

    stratify=temp_df["Label"],

    random_state=SEED

)

# ==========================================================
# DISPLAY DATASET SIZE
# ==========================================================

print("="*60)
print("DATASET SPLIT")
print("="*60)

print("Training Images   :", len(train_df))
print("Validation Images :", len(val_df))
print("Testing Images    :", len(test_df))

# ==========================================================
# DISPLAY CLASS DISTRIBUTION
# ==========================================================

print("\nTraining Distribution")

print(train_df["Label"].value_counts())

print("\nValidation Distribution")

print(val_df["Label"].value_counts())

print("\nTesting Distribution")

print(test_df["Label"].value_counts())

# ==========================================================
# RESET INDEX
# ==========================================================

train_df = train_df.reset_index(drop=True)

val_df = val_df.reset_index(drop=True)

test_df = test_df.reset_index(drop=True)

print("\nIndexes Reset Successfully.")

# ==========================================================
# VERIFY DATASET
# ==========================================================

print("="*60)
print("VERIFYING DATASET")
print("="*60)

print("Training :", train_df.shape)

print("Validation :", val_df.shape)

print("Testing :", test_df.shape)

print("\nVerification Successful.")

# ==========================================================
# LABEL ENCODING
# ==========================================================

"""
Convert class names into numerical labels.

COVID              -> 0
Lung_Opacity       -> 1
Normal             -> 2
Viral Pneumonia    -> 3
"""

label_encoder = LabelEncoder()

train_df["Label_ID"] = label_encoder.fit_transform(train_df["Label"])
val_df["Label_ID"] = label_encoder.transform(val_df["Label"])
test_df["Label_ID"] = label_encoder.transform(test_df["Label"])

print("=" * 60)
print("LABEL ENCODING")
print("=" * 60)

label_mapping = dict(
    zip(
        label_encoder.classes_,
        label_encoder.transform(label_encoder.classes_)
    )
)

print(label_mapping)

# ==========================================================
# COMPUTE CLASS WEIGHTS
# ==========================================================

"""
Class weights help compensate for any slight imbalance
remaining after dataset splitting.
"""

weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_df["Label_ID"]),
    y=train_df["Label_ID"]
)

class_weights = {
    int(i): float(w)
    for i, w in enumerate(weights)
}

print("\nClass Weights\n")

for key, value in class_weights.items():
    print(f"Class {key} : {value:.4f}")

# ==========================================================
# CREATE OUTPUT DIRECTORY
# ==========================================================

PROCESSED_PATH = os.path.join(
    PROJECT_PATH,
    "processed"
)

os.makedirs(PROCESSED_PATH, exist_ok=True)

# ==========================================================
# SAVE TRAIN / VALIDATION / TEST CSV FILES
# ==========================================================

train_df.to_csv(
    os.path.join(PROCESSED_PATH, "train.csv"),
    index=False
)

val_df.to_csv(
    os.path.join(PROCESSED_PATH, "validation.csv"),
    index=False
)

test_df.to_csv(
    os.path.join(PROCESSED_PATH, "test.csv"),
    index=False
)

print("CSV files saved successfully.")

# ==========================================================
# SAVE LABEL ENCODER
# ==========================================================

with open(
    os.path.join(PROCESSED_PATH, "label_encoder.pkl"),
    "wb"
) as file:

    pickle.dump(label_encoder, file)

print("Label Encoder saved.")

# ==========================================================
# SAVE CLASS WEIGHTS
# ==========================================================

np.save(
    os.path.join(PROCESSED_PATH, "class_weights.npy"),
    class_weights
)

print("Class Weights saved.")

# ==========================================================
# CREATE TENSORFLOW DATA PIPELINE
# ==========================================================

"""
This section creates efficient TensorFlow datasets for
training, validation and testing.

Steps:
1. Read image
2. Decode PNG
3. Resize to 224 x 224
4. Normalize pixel values
5. Batch
6. Prefetch
"""

AUTOTUNE = tf.data.AUTOTUNE

# ==========================================================
# IMAGE PREPROCESSING FUNCTION
# ==========================================================

def preprocess_image(image_path, label):

    image = tf.io.read_file(image_path)

    image = tf.image.decode_png(image, channels=3)

    image = tf.image.resize(image, IMAGE_SIZE)

    image = tf.cast(image, tf.float32) / 255.0

    return image, label

# ==========================================================
# DATASET CREATION FUNCTION
# ==========================================================

def create_dataset(dataframe, shuffle=False):

    dataset = tf.data.Dataset.from_tensor_slices(

        (
            dataframe["Image"].values,
            dataframe["Label_ID"].values
        )

    )

    dataset = dataset.map(
        preprocess_image,
        num_parallel_calls=AUTOTUNE
    )

    if shuffle:

        dataset = dataset.shuffle(
            buffer_size=1000,
            seed=SEED
        )

    dataset = dataset.batch(BATCH_SIZE)

    dataset = dataset.prefetch(AUTOTUNE)

    return dataset

# ==========================================================
# CREATE TRAIN / VALIDATION / TEST DATASETS
# ==========================================================

train_dataset = create_dataset(
    train_df,
    shuffle=True
)

validation_dataset = create_dataset(
    val_df
)

test_dataset = create_dataset(
    test_df
)

print("TensorFlow Datasets Created Successfully.")

# ==========================================================
# VERIFY ONE BATCH
# ==========================================================

print("="*60)
print("VERIFY DATASET")
print("="*60)

for images, labels in train_dataset.take(1):

    print("Image Batch Shape :", images.shape)

    print("Label Batch Shape :", labels.shape)

    print("Minimum Pixel Value :", tf.reduce_min(images).numpy())

    print("Maximum Pixel Value :", tf.reduce_max(images).numpy())

# ==========================================================
# VERIFY SAVED FILES
# ==========================================================

print("="*60)
print("VERIFYING SAVED FILES")
print("="*60)

files = [
    "train.csv",
    "validation.csv",
    "test.csv",
    "label_encoder.pkl",
    "class_weights.npy"
]

for file in files:

    path = os.path.join(PROCESSED_PATH, file)

    if os.path.exists(path):
        print(f"✓ {file}")
    else:
        print(f"✗ {file}")

# ==========================================================
# DISPLAY FINAL CLASS DISTRIBUTION
# ==========================================================

print("="*60)
print("FINAL DATASET DISTRIBUTION")
print("="*60)

print("\nTraining")
print(train_df["Label"].value_counts())

print("\nValidation")
print(val_df["Label"].value_counts())

print("\nTesting")
print(test_df["Label"].value_counts())

# ==========================================================
# DISPLAY LABEL MAPPING
# ==========================================================

print("="*60)
print("LABEL MAPPING")
print("="*60)

for class_name, label in label_mapping.items():
    print(f"{class_name:20s} --> {label}")

# ==========================================================
# DISPLAY CLASS WEIGHTS
# ==========================================================

print("="*60)
print("CLASS WEIGHTS")
print("="*60)

for key, value in class_weights.items():
    print(f"Class {key} : {value:.4f}")

# ==========================================================
# NOTEBOOK SUMMARY
# ==========================================================

print("="*60)
print("NOTEBOOK 02 COMPLETED SUCCESSFULLY")
print("="*60)

print(f"Total Images       : {len(df)}")
print(f"Training Images    : {len(train_df)}")
print(f"Validation Images  : {len(val_df)}")
print(f"Testing Images     : {len(test_df)}")

print("\nImage Size         :", IMAGE_SIZE)
print("Batch Size         :", BATCH_SIZE)

print("\nProcessed files saved in:")
print(PROCESSED_PATH)

print("\nReady for Notebook 03 - Custom CNN Model Training")



TensorFlow Version : 2.20.0
GPU Available : []
Mounted at /content/drive

Extracting Dataset...
Please wait...

Dataset Extracted Successfully.
/content/COVID19_Dataset/COVID-19_Radiography_Dataset
CREATING BALANCED DATASET
COVID                : 1000 images selected
Normal               : 1000 images selected
Lung_Opacity         : 1000 images selected
Viral Pneumonia      : 1000 images selected

Dataset Created Successfully.

                                               Image  Label
0  /content/COVID19_Dataset/COVID-19_Radiography_...  COVID
1  /content/COVID19_Dataset/COVID-19_Radiography_...  COVID
2  /content/COVID19_Dataset/COVID-19_Radiography_...  COVID
3  /content/COVID19_Dataset/COVID-19_Radiography_...  COVID
4  /content/COVID19_Dataset/COVID-19_Radiography_...  COVID

Class Distribution

Label
COVID              1000
Normal             1000
Lung_Opacity       1000
Viral Pneumonia    1000
Name: count, dtype: int64

Total Images : 4000

Dataset Shuffled Successfully.
      